**SfM BATHYMETRY WORKFLOW**

**Getting started**

To ensure working environment works properly, set to the top level of repository

In [1]:
cd ..

d:\PROGRAM\PYTHON\sfm-bathy-mapper


**Load Packages**

Firstly, we need to import packages that will be use in this notebook

In [2]:
import pandas as pd
import numpy as np


from sfmbathy.prep import load_las,tide_calc
from sfmbathy.process import ifov_calculation, visible_points, iter_camera_results, to_dataframe, process_refraction, export_pc

**Set Input and Output Directory**

Set input and output folder (Please be aware with folder name)

In [3]:
input =r"D:\PROGRAM\PROGRAM_MATLAB\GUI_SFM\BathymetricExtractionFromSfM\for_redistribution_files_only\Data_Example\LAS_KELAPA_DUA.las"
output = r"D:\PROGRAM\PYTHON\LAS_KELAPA_DUA_corrected.las"
io_data =r"D:\PROGRAM\PROGRAM_MATLAB\GUI_SFM\BathymetricExtractionFromSfM\for_redistribution_files_only\Data_Example\IO_DATA.csv"
eo_data =r"D:\PROGRAM\PROGRAM_MATLAB\GUI_SFM\BathymetricExtractionFromSfM\for_redistribution_files_only\Data_Example\EO_DATA.csv"

**Start Load Data**

The input for this process is the point cloud from SfM Photogrammetry in LAS format

In [4]:
pc,las,lon_med,lat_med = load_las(input)

CRS EPSG:32748 → EPSG:4326 | Median Lon: 106.566552, Lat: -5.648955


In [5]:
mean_elev= np.mean(pc[:,2])
mean_elev

np.float32(18.067947)

**Set Parameter for Tide Modelling**

In [6]:
tide_dir=r"D:\tide_models"
model="INATIDES"
x = lon_med
y = lat_med
start_time = "2021-08-29 3:22:00" # In UTC time, Start time for tide predictions (in the format "YYYY-MM-DD HH:MM:SS") 
end_time = "2021-08-29 3:56:00" # In UTC time, End time for tide predictions (in the format "YYYY-MM-DD HH:MM:SS") 
freq = "10min" # Frequency of tide predictions (e.g., '1H' for hourly) and '10min' for every 10 minutes

# Refractive index of water
n_water = "default" # You can specify a custom value for the refractive index of water, or use "default" to use the standard value (approximately 1.33)


In [7]:
wl = tide_calc(tide_dir, model, x, y, start_time, end_time, freq)

Modelling tides with INATIDES
Location Lon: 106.566552, Lat: -5.648955 | time range: 2021-08-29 3:22:00 to 2021-08-29 3:56:00
Representative Water Level: 0.162 m , refer to MSL
Representative Water Level: 19.086 m , refer to Ellipsoid Reference


### Actual Depth Correction Using Multi-View Stereo Photogrammetry Approach

This approach will calculate based points cloud (pc) visible on each cameras.


The camera’s instantaneous field of view (IFOV) are calculated based on the exterior orientation parameters
(x, y,z, pitch, roll, and yaw) as well as the camera’s internal parameters (focal length and sensor size).

In [8]:
fp = ifov_calculation(eo_data, io_data, mean_elev, chunk_size=1000, verbose=True)

Processing 529 cameras (chunk=1000, jobs=1)...


  529 / 529 Valid cameras processed...
Finished 529 footprint valid, 0 NaN (critical pitch).


Visibility check of each point on each camera

In [9]:
eo = pd.read_csv(eo_data)
r_sparse = visible_points(eo, fp, pc, verbose=True)

N_pt=4,437,821  N_cam=529  (dense seria 18.8 GB — menggunakan sparse)
Visible: 232,182 / 2,347,607,309 pasangan (0.0099%)
Memory sparse: 2.8 MB  (vs 18.8 GB dense)


In [11]:
for ci, pt_idxs, r_vals in iter_camera_results(r_sparse):
    print(f"Camera {ci}: {len(pt_idxs)} visible points")

df = to_dataframe(r_sparse)

r_sparse

Camera 159: 5 visible points
Camera 160: 194 visible points
Camera 161: 941 visible points
Camera 162: 1329 visible points
Camera 163: 1583 visible points
Camera 164: 1466 visible points
Camera 165: 1482 visible points
Camera 166: 1654 visible points
Camera 167: 1584 visible points
Camera 168: 1328 visible points
Camera 169: 243 visible points
Camera 177: 123 visible points
Camera 178: 118 visible points
Camera 179: 384 visible points
Camera 180: 928 visible points
Camera 181: 938 visible points
Camera 182: 185 visible points
Camera 183: 390 visible points
Camera 184: 16 visible points
Camera 186: 67 visible points
Camera 187: 615 visible points
Camera 188: 130 visible points
Camera 211: 5 visible points
Camera 217: 105 visible points
Camera 219: 18 visible points
Camera 232: 41 visible points
Camera 234: 16 visible points
Camera 284: 76 visible points
Camera 285: 650 visible points
Camera 286: 1091 visible points
Camera 287: 1257 visible points
Camera 288: 1471 visible points
Camera 2

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 232182 stored elements and shape (4437821, 529)>

Running depth correction using process_refraction function

In [12]:
pc_corrected = process_refraction(r_sparse, pc, wl, n_water)

UnboundLocalError: cannot access local variable 'pc_filtered' where it is not associated with a value

In [ ]:
pc_corrected = process_pc(pc, WL, n_water)

Number of points below water level: 4433883, percentage: 99.91%
Number of points above water level: 3938, percentage: 0.09%
Original point cloud size: 4437821, Corrected point cloud size: 4437821


In [ ]:
export_pc(pc_corrected, las, output)

Corrected LAS file saved to: D:\PROGRAM\PYTHON\LAS_KELAPA_DUA_corrected.las
